# 04 — Evaluation

Load best checkpoint, generate DDIM samples for every `(cluster_id × day_type)` combination, and run the full metric suite.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import jax
import jax.numpy as jnp

from src.data.loader    import load_raw, compute_stats, normalize, denormalize
from src.data.dataset   import make_windows, train_val_split, numpy_dataloader
from src.models.transformer1d import DiffusionTransformer1D
from src.models.diffusion     import DiffusionProcess
from src.training.train       import Trainer
from src.evaluation.metrics   import run_all_metrics

plt.rcParams['figure.dpi'] = 110
GUIDANCE_SCALE = 1.5
N_SAMPLES      = 100    # synthetic samples per condition
print('JAX devices:', jax.devices())

## 1. Reload data & cluster labels

In [ ]:
df = load_raw('../data/power.pk')
clusters_df   = pd.read_csv('../data/clusters.csv')
cluster_labels = clusters_df['cluster_id'].values
N_CLUSTERS     = int(cluster_labels.max()) + 1

timestamps = pd.date_range('2010-01-04', periods=df.shape[0], freq='15min')
if isinstance(df.index, pd.DatetimeIndex):
    timestamps = df.index

stats   = compute_stats(df, cluster_labels)
df_norm = normalize(df, stats, cluster_labels)

xs, cs, mid = make_windows(df_norm, cluster_labels, timestamps)
_, _, x_val, c_val = train_val_split(xs, cs, mid, n_meters=df.shape[1])
print(f'Val windows: {x_val.shape[0]}')

## 2. Load checkpoint

In [ ]:
key = jax.random.PRNGKey(99)

model = DiffusionTransformer1D(
    seq_len=96, d_model=128, n_heads=4, n_layers=4, d_ff=256,
    n_clusters=N_CLUSTERS, n_day_types=2, key=key,
)
diffusion = DiffusionProcess(T=1000, freq_loss_weight=0.05)
trainer   = Trainer(model, diffusion)
trainer.load('../checkpoints/best_model.pkl')
print('Checkpoint loaded.')

## 3. Generate samples per condition

In [ ]:
conditions = [
    (cid, dt, 'Weekday' if dt == 0 else 'Weekend')
    for cid in range(N_CLUSTERS)
    for dt in range(2)
]

results = {}     # condition label → {'real': ndarray, 'synthetic': ndarray}

for cid, dt, day_name in conditions:
    label = f'cluster{cid}_{day_name.lower()}'
    
    # Select matching real windows
    mask = (c_val[:, 0] == cid) & (c_val[:, 1] == dt)
    real_windows = x_val[mask]           # (n_real, 96)
    if real_windows.shape[0] == 0:
        print(f'  [SKIP] {label}: no real windows found')
        continue
    
    # Generate
    c_batch = jnp.array([[cid, dt]] * N_SAMPLES)
    gen_key = jax.random.PRNGKey(cid * 100 + dt)
    synth = diffusion.ddim_sample(
        trainer.model, c_batch,
        seq_len=96, batch_size=N_SAMPLES,
        key=gen_key, n_steps=50, guidance_scale=GUIDANCE_SCALE
    )
    synth = np.array(synth)             # (N_SAMPLES, 96)
    
    results[label] = {'real': real_windows, 'synthetic': synth, 'cluster': cid, 'day_type': dt}
    print(f'  {label}: {real_windows.shape[0]} real | {N_SAMPLES} synthetic')

## 4. Run metrics for each condition

In [ ]:
all_metric_rows = []

for label, data in results.items():
    print(f'\n=== {label} ===')
    m = run_all_metrics(data['real'], data['synthetic'], label=label)
    m['condition'] = label
    all_metric_rows.append(m)

metrics_df = pd.DataFrame(all_metric_rows).set_index('condition')
print('\n--- Summary ---')
print(metrics_df.to_string())

## 5. Summary table

In [ ]:
# Highlight key columns
key_cols = [c for c in ['acf_l2', 'crps', 'discriminative_acc'] if c in metrics_df.columns]
print('\nKey metrics:')
print(metrics_df[key_cols].round(4).to_string())

# Discriminative score should be close to 0.5 (indistinguishable from real)
if 'discriminative_acc' in metrics_df.columns:
    fig, ax = plt.subplots(figsize=(8, 3))
    metrics_df['discriminative_acc'].plot(kind='barh', ax=ax, color='steelblue')
    ax.axvline(0.5, linestyle='--', color='red', label='Ideal (0.5)')
    ax.set_xlabel('Accuracy')
    ax.set_title('Discriminative score per condition (target ≈ 0.5)')
    ax.legend(); plt.tight_layout(); plt.show()

## 6. Denormalised visual comparison

In [ ]:
hours = np.arange(96) / 4

fig, axes = plt.subplots(N_CLUSTERS, 2, figsize=(13, 4 * N_CLUSTERS))

for cid in range(N_CLUSTERS):
    cid_stats = stats[cid]
    for dt, day_name in enumerate(['Weekday', 'Weekend']):
        label = f'cluster{cid}_{day_name.lower()}'
        if label not in results:
            continue
        data = results[label]
        
        real_dn  = denormalize(data['real'],      cid, stats)  # back to original scale
        synth_dn = denormalize(data['synthetic'],  cid, stats)
        
        ax = axes[cid, dt]
        mu_r, s_r = real_dn.mean(0),  real_dn.std(0)
        mu_s, s_s = synth_dn.mean(0), synth_dn.std(0)
        
        ax.fill_between(hours, mu_r - s_r, mu_r + s_r, alpha=0.25, color='steelblue', label='Real ±σ')
        ax.fill_between(hours, mu_s - s_s, mu_s + s_s, alpha=0.25, color='coral',     label='Synth ±σ')
        ax.plot(hours, mu_r, color='steelblue', linewidth=2)
        ax.plot(hours, mu_s, color='coral',     linewidth=2, linestyle='--')
        ax.set_title(f'Cluster {cid} — {day_name}')
        ax.set_xlabel('Hour'); ax.set_ylabel('kWh / 15 min')
        ax.legend(fontsize=7)

plt.suptitle('Real vs Synthetic daily profiles (original scale)', fontsize=12)
plt.tight_layout(); plt.show()

## 7. Export metrics CSV

In [ ]:
metrics_df.to_csv('../data/evaluation_metrics.csv')
print('Saved → ../data/evaluation_metrics.csv')